# Monitoring — track calls, tokens, cost, latency and failures

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dhelamay/CPUT_ICAILY_AI2026_MaganLLM_WORKSHOP/blob/main/SA_LIB_WORKSHOP_Day2/08_monitoring/08_monitoring.ipynb)

**SA-LIB AI Workshop 2026 — Day 2: AI Agents**

Three free ways to watch an agent: a **DIY tracker** (SQLite + JSON log, no packages), **Arize Phoenix** (tracing UI, local, no account), **Langfuse** (open source; free cloud or self-hosted) and **LangSmith** (LangChain's hosted platform, free Developer plan). The agent code doesn't change; the tools only watch it.

Same example in every framework: **(1) first agent → (2) tools → (3) memory → (4) fallback → (5) multi-agent → complete agent**.

## 0. Setup
Install the packages (takes ~1 minute in Colab).

In [ ]:
%pip install -q "openai" "python-dotenv" "ddgs" "langgraph>=1.0" "langchain>=1.0" "langchain-openai>=1.0" "arize-phoenix" "openinference-instrumentation-langchain" "openinference-instrumentation-openai" "langfuse>=3" "langsmith"

### API keys
* **Colab:** click the 🔑 *Secrets* icon on the left, add e.g. `GROQ_API_KEY`, and enable notebook access.
* **VS Code / Jupyter / Linux:** put your keys in the `.env` file in the repo root (copy `.env.example`).
* Otherwise you'll be asked to paste a key below.

Choose the model provider with `LLM_PROVIDER`: `groq` · `gemini` · `openai` · `deepseek` · `openrouter` · `ollama`.

In [ ]:
import os, getpass
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))          # VS Code / Linux: read ../.env
try:                                           # Colab: read the Secrets panel
    from google.colab import userdata
    for key in ['GROQ_API_KEY', 'GOOGLE_API_KEY', 'OPENAI_API_KEY', 'DEEPSEEK_API_KEY', 'OPENROUTER_API_KEY', 'LANGSMITH_API_KEY', 'LANGFUSE_PUBLIC_KEY', 'LANGFUSE_SECRET_KEY', 'LANGFUSE_HOST', 'LLM_PROVIDER', 'LLM_MODEL']:
        try:
            os.environ[key] = userdata.get(key)
        except Exception:
            pass
except ImportError:
    pass

os.environ.setdefault("LLM_PROVIDER", "groq")   # <- change the provider here if you like
provider = os.environ["LLM_PROVIDER"]
key_name = {"groq": "GROQ_API_KEY", "gemini": "GOOGLE_API_KEY", "openai": "OPENAI_API_KEY",
            "deepseek": "DEEPSEEK_API_KEY", "openrouter": "OPENROUTER_API_KEY"}.get(provider)
if key_name and not os.getenv(key_name):
    os.environ[key_name] = getpass.getpass(f"{key_name}: ")
print("provider:", provider)

### Shared helper: `workshop_common.py`
Reads the provider settings, and holds the two tools every example uses: `web_search` (free DuckDuckGo) and `word_count`.

In [ ]:
%%writefile workshop_common.py
"""
workshop_common.py — shared helpers for every framework folder in this workshop.

(the SAME file is copied into each folder so every folder works on its own —
 edit shared/workshop_common.py and run `python tools/sync_common.py` to update the copies)

What it gives you:
  get_llm_config()   -> which provider / model / base_url / api key to use (read from .env)
  fallback_order()   -> the list of providers to try, in order, for the fallback examples
  ping()             -> quick "are you alive?" check against a provider
  web_search()       -> free DuckDuckGo search (no key)
  word_count()       -> the tiny "tool you wrote yourself" from the original guide

Every provider below speaks the OpenAI chat-completions protocol, which is why every
framework in this workshop can use every provider: we just change base_url + api_key + model.
"""

from __future__ import annotations

import os
from dataclasses import dataclass

from dotenv import find_dotenv, load_dotenv

# walks up from where you run python, so the .env in the repo root is always found
load_dotenv(find_dotenv(usecwd=True))

PROVIDERS = {
    #  name        base_url                                                     api-key env var       default model
    "groq":       ("https://api.groq.com/openai/v1",                           "GROQ_API_KEY",       "llama-3.3-70b-versatile"),
    "gemini":     ("https://generativelanguage.googleapis.com/v1beta/openai/", "GOOGLE_API_KEY",     "gemini-2.5-flash"),
    "openai":     ("https://api.openai.com/v1",                                "OPENAI_API_KEY",     "gpt-4o-mini"),
    "deepseek":   ("https://api.deepseek.com",                                 "DEEPSEEK_API_KEY",   "deepseek-chat"),
    "openrouter": ("https://openrouter.ai/api/v1",                             "OPENROUTER_API_KEY", "meta-llama/llama-3.3-70b-instruct:free"),
    "ollama":     (os.getenv("OLLAMA_BASE_URL", "http://localhost:11434/v1"),  None,                 "qwen2.5:7b"),
}


@dataclass
class LLMConfig:
    provider: str
    model: str
    base_url: str
    api_key: str

    @property
    def litellm_model(self) -> str:
        """Model name in LiteLLM format (used by CrewAI and Google ADK): 'openai/<model>'
        means 'an OpenAI-compatible endpoint' — combined with base_url it works for every provider."""
        return f"openai/{self.model}"


def get_llm_config(provider: str | None = None) -> LLMConfig:
    """Read LLM_PROVIDER / LLM_MODEL / <PROVIDER>_API_KEY from the environment."""
    provider = (provider or os.getenv("LLM_PROVIDER") or "groq").strip().lower()
    if provider not in PROVIDERS:
        raise ValueError(f"Unknown LLM_PROVIDER '{provider}'. Choose one of: {', '.join(PROVIDERS)}")
    base_url, key_env, default_model = PROVIDERS[provider]
    api_key = os.getenv(key_env) if key_env else "ollama"  # ollama runs locally, no key
    if not api_key:
        raise RuntimeError(f"{key_env} is missing — add it to your .env file (see .env.example).")
    # LLM_MODEL only overrides the model of the MAIN provider, not of the fallbacks
    main_provider = (os.getenv("LLM_PROVIDER") or "groq").strip().lower()
    model = (os.getenv("LLM_MODEL") if provider == main_provider else None) or default_model
    return LLMConfig(provider, model, base_url, api_key)


def fallback_order() -> list[str]:
    """Providers to try, in order. Default: your LLM_PROVIDER first, then LLM_FALLBACKS (groq,gemini)."""
    main = (os.getenv("LLM_PROVIDER") or "groq").strip().lower()
    extra = [p.strip().lower() for p in os.getenv("LLM_FALLBACKS", "groq,gemini").split(",") if p.strip()]
    return list(dict.fromkeys([main, *extra]))  # remove duplicates, keep order


def ping(cfg: LLMConfig) -> None:
    """Send a 1-token request. Raises an exception if the provider is down, rate-limited or the key is wrong."""
    from openai import OpenAI

    OpenAI(base_url=cfg.base_url, api_key=cfg.api_key, timeout=20, max_retries=0).chat.completions.create(
        model=cfg.model, messages=[{"role": "user", "content": "ping"}], max_tokens=1
    )


def web_search(query: str) -> str:
    """Search the web with DuckDuckGo (free, no key) and return the top results as text."""
    try:
        from ddgs import DDGS

        results = DDGS().text(query, max_results=3)
        return "\n".join(f"- {r['title']}: {r['body']} ({r['href']})" for r in results) or "no results"
    except Exception as e:  # the free search is rate-limited: return the error instead of crashing
        return f"web search failed ({e}). try again in a few seconds."


def word_count(text: str) -> int:
    """Count how many words are in a piece of text."""
    return len(text.split())

### helper file: `monitor.py`

In [ ]:
%%writefile monitor.py
"""
monitor.py — a tiny agent monitor you can read in 5 minutes. no extra packages.

It records every LLM call, tool call and agent run into a SQLite file (agent_monitor.db):
    when · which run · what kind · name/model · ok or error · error text · latency · tokens · cost
and also writes one JSON line per event to agent_events.log (easy to ship to any log system).

Use it like this:
    with monitor.run("answer a question") as run_id:        # groups everything below into one run
        resp = monitor.chat(client, model=..., messages=...)  # instead of client.chat.completions.create
        result = monitor.tool(web_search, query="...")        # instead of web_search(query="...")

Then look at the numbers:  python 08_monitoring/02_usage_report.py
"""

from __future__ import annotations

import contextvars
import json
import logging
import sqlite3
import time
import uuid
from contextlib import contextmanager
from datetime import datetime, timezone

DB_PATH = "agent_monitor.db"
LOG_PATH = "agent_events.log"

# price per 1 MILLION tokens in USD: (input, output). EXAMPLE values — check your provider's pricing page.
PRICES = {
    "gpt-4o-mini": (0.15, 0.60),
    "deepseek-chat": (0.28, 0.42),
    "gemini-2.5-flash": (0.30, 2.50),
    "llama-3.3-70b-versatile": (0.0, 0.0),   # groq free tier
}

_current_run = contextvars.ContextVar("current_run", default=None)

logger = logging.getLogger("agent_monitor")
if not logger.handlers:
    handler = logging.FileHandler(LOG_PATH)
    handler.setFormatter(logging.Formatter("%(message)s"))
    logger.addHandler(handler)
    logger.setLevel(logging.INFO)


def _db() -> sqlite3.Connection:
    con = sqlite3.connect(DB_PATH)
    con.execute("""CREATE TABLE IF NOT EXISTS events (
        ts TEXT, run_id TEXT, kind TEXT, name TEXT, status TEXT, error TEXT,
        latency_ms REAL, prompt_tokens INTEGER, completion_tokens INTEGER, cost_usd REAL)""")
    return con


def record(kind: str, name: str, status: str, latency_ms: float, error: str = "",
           prompt_tokens: int = 0, completion_tokens: int = 0) -> None:
    """Save one event to SQLite and to the JSON log."""
    price_in, price_out = PRICES.get(name, (0.0, 0.0))
    cost = (prompt_tokens * price_in + completion_tokens * price_out) / 1_000_000
    event = {"ts": datetime.now(timezone.utc).isoformat(timespec="seconds"), "run_id": _current_run.get(),
             "kind": kind, "name": name, "status": status, "error": error[:300], "latency_ms": round(latency_ms, 1),
             "prompt_tokens": prompt_tokens, "completion_tokens": completion_tokens, "cost_usd": round(cost, 6)}
    with _db() as con:
        con.execute("INSERT INTO events VALUES (:ts,:run_id,:kind,:name,:status,:error,:latency_ms,"
                    ":prompt_tokens,:completion_tokens,:cost_usd)", event)
    logger.info(json.dumps(event))


@contextmanager
def run(description: str = "agent run"):
    """Group all calls inside the `with` block into one run (and time the whole run)."""
    run_id = uuid.uuid4().hex[:8]
    token = _current_run.set(run_id)
    start = time.perf_counter()
    try:
        yield run_id
        record("run", description, "ok", (time.perf_counter() - start) * 1000)
    except Exception as e:
        record("run", description, "error", (time.perf_counter() - start) * 1000, error=repr(e))
        raise
    finally:
        _current_run.reset(token)


def chat(client, **kwargs):
    """Call client.chat.completions.create(**kwargs) and record latency, tokens, cost and errors."""
    start = time.perf_counter()
    try:
        resp = client.chat.completions.create(**kwargs)
    except Exception as e:
        record("llm", kwargs.get("model", "?"), "error", (time.perf_counter() - start) * 1000, error=repr(e))
        raise
    usage = resp.usage
    record("llm", kwargs.get("model", "?"), "ok", (time.perf_counter() - start) * 1000,
           prompt_tokens=getattr(usage, "prompt_tokens", 0) or 0,
           completion_tokens=getattr(usage, "completion_tokens", 0) or 0)
    return resp


def tool(fn, **kwargs):
    """Call a tool function and record latency and errors. Errors are returned as text (the agent keeps going)."""
    start = time.perf_counter()
    try:
        result = fn(**kwargs)
        failed = isinstance(result, str) and result.startswith("web search failed")
        record("tool", fn.__name__, "error" if failed else "ok", (time.perf_counter() - start) * 1000,
               error=result if failed else "")
        return result
    except Exception as e:
        record("tool", fn.__name__, "error", (time.perf_counter() - start) * 1000, error=repr(e))
        return f"tool error: {e}"

### helper file: `agent_graph.py`
the complete LangGraph agent (tools + memory), shared by the Phoenix and Langfuse examples.
monitoring tools don't change your agent — they only watch it.

In [ ]:
%%writefile agent_graph.py
# the complete LangGraph agent (tools + memory), shared by the Phoenix and Langfuse examples.
# monitoring tools don't change your agent — they only watch it.

from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition

import workshop_common as wc


@tool
def web_search(query: str) -> str:
    """Search the web for up-to-date information."""
    return wc.web_search(query)


@tool
def word_count(text: str) -> int:
    """Count how many words are in a piece of text."""
    return len(text.split())


def build_agent():
    cfg = wc.get_llm_config()
    model = ChatOpenAI(model=cfg.model, base_url=cfg.base_url, api_key=cfg.api_key, temperature=0)
    tools = [web_search, word_count]
    llm = model.bind_tools(tools)

    def agent_node(state: MessagesState):
        return {"messages": [llm.invoke(state["messages"])]}

    graph = StateGraph(MessagesState)
    graph.add_node("agent", agent_node)
    graph.add_node("tools", ToolNode(tools))
    graph.add_edge(START, "agent")
    graph.add_conditional_edges("agent", tools_condition)
    graph.add_edge("tools", "agent")
    return graph.compile(checkpointer=InMemorySaver())


QUESTIONS = ["my name is m0h. search for the latest langchain version, then tell me how many words your answer is.",
             "what's my name?"]

## 1. `01_simple_monitor.py`

Monitoring 01 — track your agent yourself (no extra packages, no account)
the same agent as always (web_search + word_count), but every LLM call, tool call and run
goes through monitor.py, which writes it to SQLite + a JSON log.
we also run one question with a BROKEN tool, so you can see failure tracking.

then:  python 08_monitoring/02_usage_report.py

In [ ]:
import json

from openai import OpenAI

import monitor
import workshop_common as wc

cfg = wc.get_llm_config()
client = OpenAI(base_url=cfg.base_url, api_key=cfg.api_key)


def broken_calculator(expression: str) -> str:
    """A tool that always fails — to show how failures are tracked."""
    raise ValueError("calculator is out of batteries")


TOOLS = {"web_search": wc.web_search, "word_count": wc.word_count, "calculator": broken_calculator}
SPECS = [
    {"type": "function", "function": {"name": "web_search", "description": "Search the web for up-to-date information.",
     "parameters": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}}},
    {"type": "function", "function": {"name": "word_count", "description": "Count how many words are in a piece of text.",
     "parameters": {"type": "object", "properties": {"text": {"type": "string"}}, "required": ["text"]}}},
    {"type": "function", "function": {"name": "calculator", "description": "Evaluate an arithmetic expression.",
     "parameters": {"type": "object", "properties": {"expression": {"type": "string"}}, "required": ["expression"]}}},
]


def agent(question: str, tools: list, max_steps: int = 5) -> str:
    extra = {"tools": tools} if tools else {}          # some runs use no tools at all
    messages = [{"role": "system", "content": "Use your tools when they help answer accurately."},
                {"role": "user", "content": question}]
    for _ in range(max_steps):
        msg = monitor.chat(client, model=cfg.model, messages=messages, **extra).choices[0].message   # <- tracked
        if not msg.tool_calls:
            return msg.content
        messages.append(msg)
        for call in msg.tool_calls:
            args = json.loads(call.function.arguments or "{}")
            result = monitor.tool(TOOLS[call.function.name], **args)                                  # <- tracked
            messages.append({"role": "tool", "tool_call_id": call.id, "content": str(result)})
    return "stopped: too many steps"


questions = [
    ("normal run", "search for the latest langchain version, then tell me how many words your answer is.", SPECS),
    ("broken tool", "use the calculator to compute 18500 * 0.2", [SPECS[2]]),
    ("no tools", "explain what an AI agent is in two sentences.", []),
]
for label, question, tools in questions:
    with monitor.run(label) as run_id:
        answer = agent(question, tools)
        print(f"\n[run {run_id} · {label}]\n{answer}")

print(f"\nsaved to {monitor.DB_PATH} and {monitor.LOG_PATH}. now run: python 08_monitoring/02_usage_report.py")

## 2. `02_usage_report.py`

Monitoring 02 — the usage report: calls, tokens, cost, latency, failures
reads agent_monitor.db written by 01_simple_monitor.py (or by your own agent using monitor.py).

In [ ]:
import sqlite3

from monitor import DB_PATH

con = sqlite3.connect(DB_PATH)
q = lambda sql: con.execute(sql).fetchall()  # noqa: E731

print("=" * 64 + "\nAGENT USAGE REPORT\n" + "=" * 64)

runs, run_fail = q("SELECT COUNT(*), SUM(status='error') FROM events WHERE kind='run'")[0]
print(f"runs: {runs}   failed runs: {run_fail or 0}")

print("\n-- by kind ---------------------------------------------------")
print(f"{'kind':<6} {'calls':>6} {'errors':>7} {'fail %':>7} {'avg ms':>8} {'max ms':>8}")
for kind, calls, errors, avg_ms, max_ms in q(
        "SELECT kind, COUNT(*), SUM(status='error'), AVG(latency_ms), MAX(latency_ms) "
        "FROM events GROUP BY kind ORDER BY kind"):
    print(f"{kind:<6} {calls:>6} {errors:>7} {100 * errors / calls:>6.1f}% {avg_ms:>8.0f} {max_ms:>8.0f}")

print("\n-- LLM usage by model -----------------------------------------")
print(f"{'model':<28} {'calls':>5} {'in tok':>8} {'out tok':>8} {'cost $':>9}")
for model, calls, tin, tout, cost in q(
        "SELECT name, COUNT(*), SUM(prompt_tokens), SUM(completion_tokens), SUM(cost_usd) "
        "FROM events WHERE kind='llm' GROUP BY name"):
    print(f"{model[:28]:<28} {calls:>5} {tin:>8} {tout:>8} {cost:>9.5f}")

print("\n-- tools -------------------------------------------------------")
for name, calls, errors in q("SELECT name, COUNT(*), SUM(status='error') FROM events WHERE kind='tool' GROUP BY name"):
    print(f"{name:<20} calls={calls:<4} errors={errors}")

print("\n-- last 5 errors ------------------------------------------------")
for ts, run_id, kind, name, error in q(
        "SELECT ts, run_id, kind, name, error FROM events WHERE status='error' ORDER BY ts DESC LIMIT 5"):
    print(f"{ts} run={run_id} {kind}:{name}\n    {error[:110]}")

### Start Phoenix inside the notebook
On a laptop you can instead run `phoenix serve` in a terminal. Skip this cell if Phoenix is already running.

In [ ]:
import phoenix as px
session = px.launch_app()   # UI on port 6006 (in Colab: click the link it prints)

## 3. `03_phoenix_tracing.py`

Monitoring 03 — Arize Phoenix: open-source tracing UI that runs on YOUR machine (no account, no key)
Phoenix stores OpenTelemetry traces: every graph step, LLM call (prompt, answer, tokens, latency)
and tool call, with errors highlighted in red.

step 1 (terminal 1):  phoenix serve                  -> UI at http://localhost:6006
step 2 (terminal 2):  python 08_monitoring/03_phoenix_tracing.py
step 3: open http://localhost:6006 -> project "sa-lib-day2" -> click a trace

we wire OpenTelemetry by hand (6 lines). the same lines send traces to ANY OpenTelemetry backend —
change the endpoint to Langfuse, Jaeger, Grafana Tempo, ...
other frameworks: pip install openinference-instrumentation-<crewai | openai-agents | google-adk | openai>
and call its Instrumentor().instrument(tracer_provider=provider) the same way.

In [ ]:
from openinference.instrumentation.langchain import LangChainInstrumentor
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter
from opentelemetry.sdk.resources import Resource
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor

from agent_graph import QUESTIONS, build_agent

# --- tracing setup: where to send spans, and which project they belong to ---
provider = TracerProvider(resource=Resource({"openinference.project.name": "sa-lib-day2"}))
provider.add_span_processor(SimpleSpanProcessor(OTLPSpanExporter(endpoint="http://localhost:6006/v1/traces")))
LangChainInstrumentor().instrument(tracer_provider=provider)   # every LangChain/LangGraph step becomes a span

# --- the agent, unchanged ---
agent = build_agent()
config = {"configurable": {"thread_id": "phoenix-demo"}}
for question in QUESTIONS:
    result = agent.invoke({"messages": [{"role": "user", "content": question}]}, config)
    print(result["messages"][-1].content)

provider.shutdown()   # flush spans before exit
print("\nopen http://localhost:6006 to see the traces (project: sa-lib-day2)")

## 4. `04_langfuse_tracing.py`

Monitoring 04 — Langfuse: open-source LLM observability (traces, sessions, users, cost, scores)
two ways to run it, both free:
  a) Langfuse Cloud free tier: sign up at https://cloud.langfuse.com -> project -> API keys
  b) self-host with Docker:    git clone https://github.com/langfuse/langfuse && cd langfuse && docker compose up
                               -> http://localhost:3000, create a project, copy its keys
then put the keys in .env:
  LANGFUSE_PUBLIC_KEY=pk-lf-...   LANGFUSE_SECRET_KEY=sk-lf-...   LANGFUSE_HOST=https://cloud.langfuse.com (or http://localhost:3000)

In [ ]:
import os

from langfuse import get_client, propagate_attributes
from langfuse.langchain import CallbackHandler

import workshop_common  # noqa: F401  (loads .env)
from agent_graph import QUESTIONS, build_agent

if not os.getenv("LANGFUSE_PUBLIC_KEY"):
    raise SystemExit("add LANGFUSE_PUBLIC_KEY / LANGFUSE_SECRET_KEY / LANGFUSE_HOST to .env (see the top of this file)")

langfuse = get_client()                 # reads the LANGFUSE_* variables
handler = CallbackHandler()             # the LangChain/LangGraph hook: records every step

agent = build_agent()
config = {"configurable": {"thread_id": "langfuse-demo"}, "callbacks": [handler]}

# user_id and session_id let you filter in the Langfuse UI: "all conversations of m0h", "this chat session"
with propagate_attributes(user_id="m0h", session_id="langfuse-demo", tags=["sa-lib", "day2"]):
    for question in QUESTIONS:
        result = agent.invoke({"messages": [{"role": "user", "content": question}]}, config)
        print(result["messages"][-1].content)

langfuse.flush()                        # send everything before the script exits
print(f"\nopen {os.getenv('LANGFUSE_HOST', 'https://cloud.langfuse.com')} -> Tracing / Sessions / Users")

## 5. `05_langsmith_tracing.py`

Monitoring 05 — LangSmith: LangChain's hosted tracing and monitoring platform
free "Developer" plan (1 user, a few thousand traces per month — check https://www.langchain.com/pricing).
it is NOT open source (use Langfuse or Phoenix if you need self-hosting), but it is the easiest
option for LangChain / LangGraph: tracing turns on with environment variables, no code changes.

setup:
  1. sign up at https://smith.langchain.com  ->  Settings  ->  API Keys  ->  Create API key
  2. add to .env:
       LANGSMITH_TRACING=true
       LANGSMITH_API_KEY=lsv2_...
       LANGSMITH_PROJECT=sa-lib-day2

then: https://smith.langchain.com -> Projects -> sa-lib-day2  (traces, tokens, latency, errors, monitoring charts)

In [ ]:
import json
import os

import langsmith as ls
from langchain_core.tracers.langchain import wait_for_all_tracers
from langsmith import traceable
from langsmith.wrappers import wrap_openai
from openai import OpenAI

import workshop_common as wc  # loads .env
from agent_graph import QUESTIONS, build_agent

if not os.getenv("LANGSMITH_API_KEY"):
    raise SystemExit("add LANGSMITH_TRACING=true and LANGSMITH_API_KEY=... to .env (see the top of this file)")
PROJECT = os.getenv("LANGSMITH_PROJECT") or "sa-lib-day2"
client_ls = ls.Client()   # reads LANGSMITH_API_KEY (and LANGSMITH_ENDPOINT if you self-host)

# ---------------------------------------------------------------------------------------------
# for code WITHOUT LangChain: wrap the openai client and decorate your own functions.
# this works for agents built with any framework, or none (used in part B below).
# ---------------------------------------------------------------------------------------------
cfg = wc.get_llm_config()
client = wrap_openai(OpenAI(base_url=cfg.base_url, api_key=cfg.api_key))   # every LLM call is traced


@traceable(run_type="tool")
def word_count(text: str) -> int:
    return wc.word_count(text)


@traceable(run_type="chain", name="plain_python_agent")
def plain_agent(question: str) -> str:
    spec = [{"type": "function", "function": {"name": "word_count", "description": "Count words in a text.",
             "parameters": {"type": "object", "properties": {"text": {"type": "string"}}, "required": ["text"]}}}]
    messages = [{"role": "user", "content": question}]
    for _ in range(4):                                   # plan -> act -> observe
        msg = client.chat.completions.create(model=cfg.model, messages=messages, tools=spec).choices[0].message
        if not msg.tool_calls:
            return msg.content
        messages.append(msg)
        for call in msg.tool_calls:
            result = word_count(**json.loads(call.function.arguments or "{}"))
            messages.append({"role": "tool", "tool_call_id": call.id, "content": str(result)})
    return "stopped: too many steps"


# ---------------------------------------------------------------------------------------------
# run both. with LANGSMITH_TRACING=true in .env you don't even need this `with` block for LangGraph —
# we use it so the demo always traces (also in notebooks) and to name the project in one place.
# ---------------------------------------------------------------------------------------------
with ls.tracing_context(enabled=True, project_name=PROJECT, client=client_ls):
    # part A — LangGraph agent: NOTHING added to the agent itself. every node, LLM call and tool call is traced.
    agent = build_agent()
    config = {"configurable": {"thread_id": "langsmith-demo"},
              "metadata": {"user_id": "m0h"}, "tags": ["sa-lib", "langgraph"]}   # filterable in the UI
    for question in QUESTIONS:
        result = agent.invoke({"messages": [{"role": "user", "content": question}]}, config)
        print("[langgraph]", result["messages"][-1].content)

    # part B — the plain-python agent
    print("[plain]", plain_agent("how many words are in 'agents plan act and observe'? use the tool."))

wait_for_all_tracers()   # send LangChain/LangGraph traces before exit
client_ls.flush()        # send @traceable / wrap_openai traces before exit
print(f"\nopen https://smith.langchain.com -> Projects -> {PROJECT}")

---
*SA-LIB AI Workshop 2026 · based on [build-ai-agents-free](https://github.com/Moh4696/build-ai-agents-free) by m0h (MIT)*